# C2组（G21）第08章 人工复核

对应 `C2组-手动推导.md` 的 D1—D4。四个单元各回答一个问题，末尾有断言；数字与推导文件一致。

运行方式：`Kernel → Restart Kernel and Run All Cells`，全部为确定性计算，无随机抽样（POD 快照由解析式生成）。


In [ ]:
import platform
import numpy as np

print({"python": platform.python_version(), "numpy": np.__version__})

## D1　CG：$κ$ 相同，步数可以差一倍

1D Laplacian（$n=64$）上比较三种预条件子。第三个是“伪学习型”对角预条件：每行乘一个 $1+0.02\cos i$ 的扰动，$κ$ 几乎不变，看步数会不会变。

In [ ]:
def cg(A, b, Minv=None, tol=1e-8, maxit=5000):
    x = np.zeros_like(b)
    r = b - A @ x
    z = r if Minv is None else Minv @ r
    p = z.copy()
    rz = r @ z
    bnorm = np.linalg.norm(b)
    for k in range(1, maxit + 1):
        Ap = A @ p
        alpha = rz / (p @ Ap)
        x += alpha * p
        r -= alpha * Ap
        if np.linalg.norm(r) / bnorm < tol:
            return k
        z = r if Minv is None else Minv @ r
        rz_new = r @ z
        p = z + (rz_new / rz) * p
        rz = rz_new
    return maxit

n = 64
A = np.diag(2.0 * np.ones(n)) - np.diag(np.ones(n - 1), 1) - np.diag(np.ones(n - 1), -1)
b = np.ones(n)
D = np.diag(np.diag(A))
learned = np.diag(1.0 / (np.diag(A) * (1 + 0.02 * np.cos(np.arange(n)))))

rows = [
    ("无预条件", np.linalg.cond(A), cg(A, b)),
    ("Jacobi", np.linalg.cond(np.linalg.inv(D) @ A), cg(A, b, np.linalg.inv(D))),
    ("伪学习型对角", np.linalg.cond(learned @ A), cg(A, b, learned)),
]
print(f"{'预条件':<14}{'kappa':>10}{'迭代数':>8}")
for name, kappa, it in rows:
    print(f"{name:<14}{kappa:>10.1f}{it:>8}")

assert abs(rows[1][1] / rows[2][1] - 1) < 0.01, "两者 kappa 应几乎相同"
assert rows[2][2] >= 1.7 * rows[1][2], "扰动后步数应明显增加"
print("D1 通过：kappa 几乎相同，步数从", rows[1][2], "变为", rows[2][2])

## D2　“保留 99% 能量”对应的是 10% 的 Frobenius 相对误差

由 $\|S-S_r\|_F/\|S\|_F=\sqrt{1-E_r}$，$E_r=0.99$ 时相对误差是 $\sqrt{0.01}=10\%$。
下表用真实 SVD 检验这条恒等式，并给出最大点误差。

In [ ]:
x = np.linspace(0, 1, 200)
params = np.linspace(0.2, 1.0, 40)
S = np.array([np.exp(-a * 3.0 * x) * np.sin(np.pi * x) for a in params]).T

U, s, Vt = np.linalg.svd(S, full_matrices=False)
energy = np.cumsum(s ** 2) / np.sum(s ** 2)
print(f"{'r':>3}{'能量占比':>12}{'相对F误差':>14}{'最大点误差':>14}{'sqrt(1-E)':>13}")
for r in (1, 2, 3):
    Sr = U[:, :r] @ np.diag(s[:r]) @ Vt[:r]
    rel = np.linalg.norm(S - Sr) / np.linalg.norm(S)
    mx = np.max(np.abs(S - Sr))
    print(f"{r:>3}{energy[r-1]:>12.4f}{rel:>14.3e}{mx:>14.3e}{np.sqrt(1-energy[r-1]):>13.3e}")

assert abs(np.linalg.norm(S - U[:, :1] @ np.diag(s[:1]) @ Vt[:1]) / np.linalg.norm(S)
           - np.sqrt(1 - energy[0])) < 1e-12, "恒等式应成立"
assert energy[0] > 0.98 and np.max(np.abs(S - U[:, :1] @ np.diag(s[:1]) @ Vt[:1])) > 0.05
print("D2 通过：能量", round(float(energy[0]), 4), "最大点误差", f"{np.max(np.abs(S - U[:, :1] @ np.diag(s[:1]) @ Vt[:1])):.3f}")

## D3　自由滚动的误差上界

$e_T\le L^T e_0+\eta(L^T-1)/(L-1)$，取 $L=1.05$、$\eta=10^{-10}$。

In [ ]:
L, eta = 1.05, 1e-10
print(f"{'T':>5}{'eta*L^T':>14}{'累积上界':>14}")
for T in (10, 100, 300):
    bound = eta * (L ** T - 1) / (L - 1)
    print(f"{T:>5}{eta * L ** T:>14.3e}{bound:>14.3e}")

assert eta * (L ** 300 - 1) / (L - 1) > 1e-3, "300 步应达到 1e-3 量级"
print("D3 通过：300 步上界约", f"{eta * (L ** 300 - 1) / (L - 1):.2e}")

## D4　范围外的 POD 投影

用 $a\in[0.2,1]$ 的 40 个快照建 2 维基，再投影到 $a=1.2$ 与 $a=2$。

In [ ]:
U_tr = np.linalg.svd(S, full_matrices=False)[0][:, :2]
print(f"{'a':>5}{'相对投影误差':>16}{'最大点误差':>14}")
errs = {}
for a_test in (0.6, 1.2, 2.0):
    u = np.exp(-a_test * 3.0 * x) * np.sin(np.pi * x)
    rec = U_tr @ (U_tr.T @ u)
    errs[a_test] = (np.linalg.norm(u - rec) / np.linalg.norm(u), np.max(np.abs(u - rec)))
    print(f"{a_test:>5}{errs[a_test][0]:>16.3e}{errs[a_test][1]:>14.3e}")

assert errs[0.6][0] < 0.02, "范围内误差应小"
assert errs[2.0][0] > 0.15, "范围外误差应显著增大"
print("D4 通过：范围内", f"{errs[0.6][0]:.3f}", "范围外", f"{errs[2.0][0]:.3f}")

## 结论

四条都与推导文件的数字一致：$κ$ 只给趋势、99% 能量对应 10% 相对误差、300 步误差放大到 $10^{-3}$、范围外投影误差放大 28 倍。

**这不是说 A1/A2 写错了**——两稿都提到了这些限定；本复核的作用是把限定变成可复算的量级，供六组讨论时引用。
